In [ ]:
print('hello chart')

In [ ]:
# FILE PATH and SQL QUERY
import pandas as pd
import duckdb
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

# file path
folder = Path.cwd() / '_inputs'/'dayly_klines'
# access the files
sorted_pqt_files = sorted(folder.glob('*parquet'))
# file list
pqt_file = [pqt.as_posix() for pqt in sorted_pqt_files]

# duckdb query for the data
conn = duckdb.connect()

qry = f"""--sql
        select 
            --cast for ms to readeable time
            --epoch_ms(CAST(open_time as bigint)) as open_time,
            open_time,
            open,
            high,
            low,
            close,
            close_time
        from read_parquet({pqt_file})    
        """

df = conn.sql(qry).df()
# df = conn.execute(query).df()
conn.close()

df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')
df["sma25"] = df['close'].rolling(25).mean()

df.tail()

In [ ]:
# the SIMPLE MOVING AVERAGE
# df["sma25"] = df['close'].rolling(25).mean()

# fig1 = go.Figure()

# fig.add_trace(go.Scatter(x=df.open_time, y=df.sma25, mode='lines', line=dict(color='purple')))
# fig1.show()

In [ ]:
# THE CANDLE CHART



fig = go.Figure()
fig.add_trace(go.Scatter(x=df.open_time, y=df.sma25, mode='lines', line=dict(color='purple', width =1)))
fig.add_trace(go.Candlestick(x=df['open_time'],
                             open=df['open'],
                             high=df['high'],
                             low=df['low'],
                             close=df['close'],
                             name='OHLC',
                             # bullish candles
                             increasing_line_color='white',
                             increasing_fillcolor='rgba(0, 0, 0, 0)',
                             increasing_line_width=1,
                             # bearish candles
                             decreasing_line_color='#0677d4',
                             decreasing_fillcolor= 'rgba(0, 0, 0, 0)',
                             decreasing_line_width=1
                             ))

fig.update_layout(title= 'sol 2026 jan to june  day timeframe',
                  template='plotly_dark',
                  xaxis_rangeslider_visible=False,
                  hovermode='x unified',
                  showlegend= False,
                  height= 600
                  )
# horizontal line
h_line= dict(
     y=80,
    line_dash='dash', # Options: "solid", "dot", "dash", "dashdot"
    line_color= 'red',
    line_width=1,
    annotation_text= 'Target: 70',
    annotation_position= 'top right'
)
fig.add_hline( **h_line)
# vertical line

fig.add_vline(
    #the string given to x must match the format of your df.time column. If your time column contains full dates like '2024-01-15 03:00:00', pass the exact string timestamp or a pd.Timestamp('2026-06-20 03:00:00')
    x=pd.Timestamp('2026-04-20'),
    line_dash='dot', # Options: "solid", "dot", "dash", "dashdot"
    line_color= 'green',
    line_width=1,
    annotation_text= '3am mark',
    annotation_position= 'top right'
)

fig.show()